# OPENAI - Lab2
___

## Setup SendGrid

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, set_default_openai_client
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os 
from sendgrid.helpers.mail import Mail, Email, To, Content 
import asyncio
from loguru import logger

load_dotenv(override=True)
client = AsyncOpenAI(
    api_key = os.getenv("OPENAI_API_KEY")
)
set_default_openai_client(client)

In [ ]:
class EmailSender:
    def __init__(self, 
                api_key: str,
                email_sender: str) -> None:
        self.sg = sendgrid.SendGridAPIClient(api_key=api_key)
        self.email_sender = Email(email_sender)
    
    def send_email(self, 
                email_recipient: str,
                content: str) -> int:
        to_email = To(email_recipient)
        content_email = Content("text/plain", content)
        mail = Mail(self.email_sender, to_email, "Test email", content_email).get()
        response = self.sg.client.mail.send.post(request_body=mail)
        logger.info(f"RESPONSE STATUS CODE: {response.status_code}")
        return response.status_code

email_sender = EmailSender(api_key = os.getenv("SENDGRID_API_KEY"),
                        email_sender = os.getenv("EMAIL_SENDER"))

email_recipient = "stefanus.yudi96@gmail.com"
content = "Test Email from SenGrid App"
email_sender.send_email(email_recipient=email_recipient, content=content)


2026-05-27 20:48:12.576 | INFO     | __main__:send_email:15 - RESPONSE STATUS CODE: 202


202

## Step 1: Agent Workflow

In [6]:
OPENAI_MODEL = os.getenv("OPENAI_MODEL")
instruction1 = "You are a sales agent working for ComplAI, \
    a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
    You write professional, serious cold emails."

instruction2 = "You are a humorous, engaging sales agent working for ComplAI, \
    a company that provides a SaaS tool for ensuring SOC2 compliance an preparing for audits, powered by AI. \
    You write witty, engaging cold emails that are likely to get a response"

instruction3 = "You are a humorous, engaging sales agent working for ComplAI, \
    a company that provides a SaaS tool for ensuring SOC2 compliance an preparing for audits, powered by AI. \
    You write concise, to the point cold emails."

In [7]:
sales_agent1 = Agent(
    name = "Professional Sales Agent",
    instructions = instruction1,
    model = OPENAI_MODEL
)

sales_agent2 = Agent(
    name = "Engaging Sales Agent",
    instructions = instruction2,
    model = OPENAI_MODEL
)

sales_agent3 = Agent(
    name = "Busy Sales Agent",
    instructions = instruction3,
    model = OPENAI_MODEL
)

In [10]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC2 Compliance with AI-Powered Solutions

Dear [Recipient’s Name],

I hope this message finds you well. As organizations like yours strive to maintain rigorous security standards, SOC2 compliance remains a critical aspect of building trusted relationships with clients and partners.

At ComplAI, we offer an AI-driven SaaS platform designed to simplify and accelerate your SOC2 compliance journey. Our tool automates documentation, monitors control activities in real-time, and prepares you thoroughly for audits — all while reducing manual effort and risk of errors.

Would you be open to a brief conversation to explore how our platform can support your compliance efforts and enhance your security posture?

Thank you for your time. I look forward to the opportunity to connect.

Best regards,  
[Your Name]  
[Your Title]  
ComplAI  
[Your Contact Information]

In [13]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output+"\n\n")

Subject: Simplify Your SOC2 Compliance and Audit Preparation with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. Managing SOC2 compliance can be complex and time-consuming, but it doesn’t have to be. At ComplAI, we’ve developed an AI-powered SaaS platform specifically designed to streamline compliance tracking and prepare you effortlessly for audits.

Our solution provides real-time insights, automates documentation, and helps detect potential issues before they become obstacles—saving your team valuable time and reducing risk.

Would you be open to a quick call next week to explore how ComplAI can support your compliance efforts?

Looking forward to your response.

Best regards,  
[Your Name]  
[Your Title]  
ComplAI  
[Your Contact Information]


Subject: SOC2 Stress? Meet Your New Best Friend—ComplAI 🤖✨

Hey [First Name],

Are you tired of feeling like you need a crystal ball to predict your SOC2 audit? Trust me, you’re not alone. Navigating compliance can be a

In [14]:
sales_picker = Agent(
    name = "sales_picker",
    instructions = "You pick the best cold sales email from the given options. \
        Imagine you are a customer and pick the one you are most likely to respond to. \
        Do not give an explanation; reply with the selected email only.",
    model = OPENAI_MODEL
)

In [15]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]
    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)
    print(f"Best sales email: \n{best.final_output}")

Best sales email: 
Subject: Is SOC2 Compliance Holding You Hostage? Let's Free You Up! 🚀


## Step 2: Use of Tools

In [18]:
@function_tool
def send_email(body: str):
    """Send out an email with the given body to all sales prospects
    """
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv('SENDGRID_API_KEY'))
    from_email = Email(os.getenv("EMAIL_SENDER"))
    to_email = To("stefanus.yudi96@gmail.com")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [19]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x73f0353c6780>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [20]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x73f035399760>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [21]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name = "sales_agent1", tool_description = description)
tool2 = sales_agent2.as_tool(tool_name = "sales_agent2", tool_description = description)
tool3 = sales_agent3.as_tool(tool_name = "sales_agent3", tool_description = description)

tools = [tool1, tool2, tool3, send_email]
tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x73f0353ccd40>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Fa

In [22]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools. 

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgement of which one is most effective.

3. Use the send_email tool to send the best email (and only the best email) to the user.

Crucial Rules:
- You must use the sales agent tools to generate the drafts - do not write them yourself.
- You must send ONE email using the send_email tool - never more than one.
"""
sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=OPENAI_MODEL)
message = "Send a cold sales email addressed to 'Dear CEO'"
with trace("Sales Manager"):
    result = await Runner.run(sales_manager, message)   

In [27]:
subject_instructions = "You can write a subject for a cold sales email. \
    You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
    You are given a text email body which might have some markdown \
    and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name = "Email subject writier", instructions = subject_instructions, model = OPENAI_MODEL)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Writer a subject for a cold sales email")

html_converter = Agent(name = "HTML email body converter", instructions=html_instructions, model = OPENAI_MODEL)
html_tool = html_converter.as_tool(tool_name = "html_converter", tool_description = "Convert a text email body to an HTML email body")

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send out and email with the given subject and HTML body to all sales prospect
    """
    sg = sendgrid.SendGridAPIClient(api_key = os.getenv("SENDGRID_API_KEY"))
    from_email = Email(os.getenv("EMAIL_SENDER"))
    to_email = To("stefanus.yudi96@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

tools = [subject_tool, html_tool, send_html_email]
tools

[FunctionTool(name='subject_writer', description='Writer a subject for a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x73f01f2f5bb0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties

In [30]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model=OPENAI_MODEL,
    handoff_description="Convert an email to HTML and send it")

In [29]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x73f0353ccd40>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Fai

In [31]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=OPENAI_MODEL)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function calling and has been transformed to 'transfer_to_email_manager'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Email Manager' contains invalid characters for function c